## 1.Environment Setup

In [ ]:
# Import thư viện cần thiết
import os
import random
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings('ignore')

# Đặt seed để đảm bảo kết quả nhất quán giữa các lần chạy
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
print(f"Seed: {SEED}")

## 2.Data Processing

In [ ]:
# https://drive.google.com/drive/folders/1DzX2wNAtNlQZksk69BjS-uJaeQ6TBRgH?usp=sharing
!gdown --folder 1DzX2wNAtNlQZksk69BjS-uJaeQ6TBRgH

In [ ]:
def read_csv(file_path):
    df = pd.read_csv(file_path)
    display(df.head())

    X = df.drop('target', axis=1)
    y = df['target']
    display(y.value_counts())

    print("Shape df: ", df.shape)
    print("Shape X: ", X.shape)
    print("Shape y: ", y.shape)

    return X, y

### 2.1 Original Dataset

In [ ]:
X_train, y_train =      # your code
X_val, y_val =      # your code
X_test, y_test =      # your code

### 2.2 FE Dataset

In [ ]:
X_fe_train, y_fe_train =      # your code
X_fe_val, y_fe_val =      # your code
X_fe_test, y_fe_test =      # your code

### 2.3 Original + DT Dataset

In [ ]:
X_dt_train, y_dt_train =      # your code
X_dt_val, y_dt_val =      # your code
X_dt_test, y_dt_test =      # your code

### 2.4 FE + DT Dataset

In [ ]:
X_fe_dt_train, y_fe_dt_train =      # your code
X_fe_dt_val, y_fe_dt_val =      # your code
X_fe_dt_test, y_fe_dt_test =      # your code

## 3.AdaBoost (AB)

In [ ]:
def find_optimal_ada(
    X_train, y_train, base_max_depth=1, algorithm="SAMME"
    n_estimators_range=range(50, 501, 50),
    cv_splits=3, learning_rate=0.1,
):
    # your code
    print(f"n_estimators tối ưu (CV): {best_n}")

    # Vẽ biểu đồ chọn n_estimators theo CV
    plt.figure(figsize=(10, 6))
    plt.plot(list(n_estimators_range), scores, 'bo-')
    plt.title(f'Chọn n_estimators tối ưu cho AdaBoost (CV={cv_splits}-fold)')
    plt.xlabel('n_estimators')
    plt.ylabel('Cross-Validation Accuracy')
    plt.grid(True)
    plt.show()

    # Train lại với toàn bộ X_train
    best_model =       # your code
    best_model.fit(      # your code)
    return best_model, best_n, max(scores)

def evaluate_val_ada(X_train, y_train, X_val, y_val,
                     n_estimators_range=range(50, 501, 50),
                     cv_splits=3, learning_rate=0.1,
                     base_max_depth=1, algorithm="SAMME"):
    print("Tìm n_estimators tối ưu cho AdaBoost...")
    ada_model, best_n, cv_acc =       # your code

    # Dự đoán & báo cáo trên validation
    val_pred =       # your code
    val_acc =       # your code
    print(f"\nĐộ chính xác AdaBoost trên tập validation: {val_acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_val, val_pred))
    return ada_model, val_acc, {"n_estimators": best_n}

def evaluate_test_ada(ada_model, X_test, y_test):
    test_pred =       # your code
    test_acc =       # your code
    print(f"\nĐộ chính xác AdaBoost trên tập test: {test_acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, test_pred))
    return test_acc

### 3.1 AB on Original Dataset

In [ ]:
ada_model, val_acc, best_params =       # your code

test_acc =       # your code

### 3.2 AB on FE Dataset

In [ ]:
ada_model, val_fe_acc, best_params =      # your code

test_fe_acc =      # your code

### 3.3 AB on Original DT Dataset

In [ ]:
ada_model, val_dt_acc, best_params =       # your code

test_dt_acc =       # your code

### 3.4 AB on FE + DT Dataset

In [ ]:
ada_model, val_fe_dt_acc, best_params =       # your code

test_fe_dt_acc =       # your code

## 4.Result Visualization

In [ ]:
plt.rcParams['font.family'] = 'Serif'

labels = ['Original', 'FE', 'Original + DT', "FE + DT"]
val_accs = [val_acc, val_fe_acc, val_dt_acc, val_fe_dt_acc]
test_accs = [test_acc, test_fe_acc, test_dt_acc, test_fe_dt_acc]

x = np.arange(len(labels))
width = 0.3

fig, ax = plt.subplots(figsize=(5, 5))

rects1 = ax.bar(x - width/2, val_accs,  width,
                label='Validation Accuracy',
                color='tab:blue', edgecolor='black', linewidth=1.2)
rects2 = ax.bar(x + width/2, test_accs, width,
                label='Test Accuracy',
                color='tab:red', edgecolor='black', linewidth=1.2)

ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('AdaBoost', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(ncol=2, loc="upper center")

def autolabel(rects):
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:.2f}', xy=(rect.get_x()+rect.get_width()/2, h),
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

fig.tight_layout()
fig.savefig("accuracy_comparison.png", dpi=300, bbox_inches="tight")
fig.savefig("accuracy_comparison.pdf", bbox_inches="tight")
plt.show()
